In [ ]:
!pip install mawo-pymorphy3

import re
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()
from mawo_pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()

подготовка к преобразованию

In [18]:
# список стоп-слов
STOP_WORDS = {
    'это', 'этот', 'эта', 'эти', 'этого', 'этому', 'этим', 'этом',
    'тот', 'та', 'то', 'те', 'того', 'тому', 'тем', 'том',
    'весь', 'вся', 'все', 'всё', 'всего', 'всему', 'всем', 'всём',
    'каждый', 'каждая', 'каждое', 'каждые', 'каждого',
    'свой', 'своя', 'свое', 'свои', 'своего',
    'так', 'такой', 'такая', 'такое', 'такие',
    'очень', 'можно', 'нужно', 'надо', 'просто', 'даже',
    'ещё', 'уже', 'вот', 'тут', 'там', 'здесь',
    'куда', 'откуда', 'тогда', 'потом', 'сейчас', 'всегда',
    'быть', 'является', 'находится', 'становится',
    'который', 'которая', 'которое', 'которые', 'которого',
    'этот', 'иметь', 'мочь', 'хотеть', 'сказать', 'говорить', 'думать',
    'идти', 'прийти', 'уйти', 'пойти', 'стать', 'делать', 'знать',
    'видеть', 'смотреть', 'спросить', 'ответить', 'понять',
    'человек', 'год', 'время', 'день', 'раз', 'место', 'жизнь',
    'вообще', 'действительно', 'конечно', 'наверное', 'наконец',
    'однако', 'поэтому', 'потому', 'почему', 'зачем', 'отчего',
    'вдруг'
}

def clean_text(text):
    """Очистка текста: lowercase, удаление мусора, нормализация пробелов"""
    if not isinstance(text, str) or len(text.strip()) < 3:
        return ''

    # Lowercase
    text = text.lower()

    # Оставляем только русские буквы, ё, пробелы, и важные знаки препинания
    text = re.sub(r'[^а-яё\s]', ' ', text)

    # Нормализация пробелов
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def tokenize_and_filter(text):
    """Токенизация, удаление стоп-слов, фильтрация коротких слов"""
    if not text:
        return []

    words = text.split()

    # Фильтрация
    words = [w for w in words if len(w) >= 3]  # удаляем слова короче 3 букв
    words = [w for w in words if w not in STOP_WORDS]

    return words

def lemmatize_words(words):
    """Лемматизация списка слов"""
    if not words:
        return []

    lemmatized = []
    for word in words:
        try:
            lemma = morph.parse(word)[0].normal_form
            lemmatized.append(lemma)
        except:
            lemmatized.append(word)

    return lemmatized

def full_preprocessing(text):
    """собираем все вместе"""
    cleaned = clean_text(text)
    if not cleaned:
        return ''

    words = tokenize_and_filter(cleaned)
    if not words:
        return ''

    lemmas = lemmatize_words(words)

    return ' '.join(lemmas)

словарь для разметки

In [ ]:
ASPECT_KEYWORDS_RAW = {
    'treatment_result': [
        'помог', 'помогла', 'помогло', 'помогли',
        'вылечил', 'вылечила', 'вылечило', 'вылечили',
        'стало лучше', 'улучшение', 'улучшилось', 'результат',
        'полегчало', 'полегчала', 'полегчали', 'прошла боль',
        'эффект', 'спасибо', 'выздоровел', 'выздоровела',
        'излечился', 'излечилась', 'ремиссия', 'восстановился',
        'восстановилась', 'симптомы ушли', 'здоров', 'здорова',
        'выздоравливаю', 'положительная динамика', 'нашёл решение',
        'поставил на ноги', 'вернул качество жизни', 'избавил от проблемы',
        'назначил правильное лечение', 'терапия помогла', 'лечение эффективно',
        'доволен результатом', 'результатом доволен', 'помощь', 'всё прошло',
        'реально помог', 'ощутимый результат', 'есть прогресс', 'исцелил',
        'облегчил состояние', 'улучшилось состояние', 'стало легче', 'почувствовал облегчение'
    ],

    'communication': [
        'внимательный', 'внимательная', 'внимательно',
        'объяснил', 'объяснила', 'объяснили', 'выслушал', 'выслушала',
        'понятно объяснил', 'доверие', 'чуткий', 'отзывчивый', 'отзывчивая',
        'терпеливый', 'терпеливая', 'доброжелательный', 'доброжелательная',
        'поддержал', 'поддержала', 'успокоил', 'успокоила', 'вежливый', 'вежливая',
        'тактичный', 'деликатный', 'приятный в общении', 'расположил к себе',
        'внимание', 'культурный', 'профессионально', 'подробно всё рассказал',
        'доступно объяснил', 'на все вопросы ответил', 'индивидуальный подход',
        'грубый', 'грубая', 'грубо', 'грубил', 'грубила', 'хамил', 'хамила',
        'нахамил', 'нахамила', 'перебивал', 'перебивала', 'не слушал', 'не слушала',
        'не выслушал', 'не выслушала', 'не дослушал', 'не объяснил', 'не объяснила',
        'плохое отношение', 'нагрубил', 'нагрубила', 'холодный', 'холодная',
        'безразличный', 'безразличная', 'неприветливый', 'неприветливая',
        'разговаривал свысока', 'тыкал', 'без объяснений', 'не уважает',
        'не хочет слушать', 'отмахнулся', 'наехал', 'орал', 'кричал',
        'хам', 'грубость', 'безразличие', 'нахамить'
    ],

    'diagnostics': [
        'диагноз', 'поставил диагноз', 'диагностировал', 'диагностировала',
        'анализ', 'анализы', 'сдать анализы', 'назначил анализы',
        'мрт', 'кт', 'компьютерная томография', 'магнитно-резонансная томография',
        'назначил', 'назначила', 'назначение', 'лечение назначил',
        'план лечения', 'обследование', 'узи', 'ультразвук', 'ультразвуковое исследование',
        'хгч', 'лапароскопия', 'биопсия', 'рентген', 'фгс', 'гастроскопия',
        'колоноскопия', 'эхокг', 'экг', 'электрокардиограмма', 'эндоскопия',
        'скрининг', 'посев', 'мазок', 'кровь', 'гормоны', 'онкомаркеры',
        'осмотр', 'пальпация', 'прослушал', 'перкуссия', 'аускультация',
        'консультация', 'профилактический осмотр', 'второе мнение',
        'подготовка к беременности', 'диспансеризация', 'чекап',
        'выявил', 'обнаружил', 'нашел причину', 'осмотрел', 'проверил',
        'собрал анамнез', 'выписал рецепт', 'выписала рецепт'
    ],

    'price_value': [
        'дорого', 'дороговато', 'дёшево', 'дешево', 'дешевле', 'недорого',
        'оправдано', 'оправданно', 'не стоит', 'жаль денег', 'жалко денег',
        'за такие деньги', 'переплата', 'переплатил', 'переплатила',
        'навязал услуги', 'навязала услуги', 'навязали услуги',
        'лишние анализы', 'лишние услуги', 'трата денег', 'пустая трата',
        'выброшенные деньги', 'цена завышена', 'завышенная цена', 'ценник завышен',
        'необоснованно дорого', 'неоправданно дорого', 'целиком деньги вытянули',
        'экономия', 'адекватная цена', 'приемлемая стоимость', 'бюджетно',
        'соотношение цена качество', 'бесплатно', 'платно', 'платная консультация',
        'дополнительные услуги', 'цена вопроса', 'сэкономил', 'сэкономила',
        'денег не жалко', 'ценник', 'прайс', 'стоимость приёма'
    ],

    'service_complaints': [
        'очередь', 'большая очередь', 'долгая очередь', 'огромная очередь',
        'запись', 'записаться', 'проблема с записью', 'запись на месяц',
        'администратор', 'администраторы', 'регистратура',
        'хамство в регистратуре', 'неприветливая администрация',
        'ожидание', 'долго ждал', 'долго ждала', 'долгое ожидание',
        'не дозвониться', 'не дозвонился', 'дозвониться невозможно',
        'не отвечают по телефону', 'не берут трубку', 'автоответчик',
        'опоздание', 'пришёл не вовремя', 'задержали', 'перенесли запись',
        'отменили запись', 'бюрократия', 'волокита', 'потеряли документы',
        'ошибка в записи', 'электронная запись', 'живая очередь', 'талон'
    ],

    'red_flags': [
        'не помог', 'не помогла', 'не помогло', 'не помогли',
        'стало хуже', 'ухудшение', 'ухудшилось', 'похуже',
        'не рекомендую', 'не рекомендую никому', 'никому не советую',
        'навязал услуги', 'навязала услуги',
        'ужасный врач', 'ужасный', 'ужасная', 'ужасно', 'кошмар', 'кошмарный',
        'бесполезно', 'бесполезный', 'осложнение', 'осложнения',
        'отвратительный', 'отвратительно', 'ужас', 'катастрофа',
        'убежала', 'убежал', 'больше не пойду',
        'разочарование', 'разочарован', 'разочарована',
        'зря пришёл', 'зря пришла', 'зря потратил время',
        'врач не профессионал', 'непрофессионализм',
        'шарлатан', 'некомпетентный', 'некомпетентная',
        'опасно', 'риск для жизни', 'аллергия', 'побочные эффекты',
        'неправильный диагноз', 'ошибка в диагнозе',
        'пропустил болезнь', 'поздно обнаружил',
        'ничего не понял', 'беда', 'расстройство', 'дно'
    ]
}

In [ ]:
def lemmatize_keywords(keywords_list):
    """приведение словаря к нормальной форме"""
    lemmatized = []
    for kw in keywords_list:
        words = kw.lower().split()
        lemm_words = []
        for w in words:
            lemma = morph.parse(w)[0].normal_form
            lemm_words.append(lemma)
        lemmatized.append(' '.join(lemm_words))
    return list(set(lemmatized))

ASPECT_KEYWORDS = {}
for aspect, keywords in ASPECT_KEYWORDS_RAW.items():
    ASPECT_KEYWORDS[aspect] = lemmatize_keywords(keywords)

подготовим данные

In [ ]:
df = pd.read_csv('/content/doctors_review.txt')
del df['Unnamed: 0']

len(df)

In [ ]:
df = df[df['comment'].notna()]
df = df[df['comment'].str.strip() != '']
df = df.drop_duplicates(subset=['comment'])
len(df)

In [ ]:
df['clean_text'] = df['comment'].progress_apply(full_preprocessing)

# удаляем отзывы, которые стали пустыми
df = df[df['clean_text'] != '']

len(df)

In [ ]:
def weak_label_on_clean(text_clean, aspect_keywords):
    """
    Работает с уже очищенным и лемматизированным текстом
    """
    if not isinstance(text_clean, str) or len(text_clean.strip()) < 5:
        return {aspect: 0 for aspect in aspect_keywords.keys()}

    negation_words = {'не', 'нет', 'ни', 'без', 'вовсе', 'отнюдь', 'никак', 'нисколько'}

    labels = {}
    for aspect, keywords in aspect_keywords.items():
        found = False
        for kw in keywords:
            if kw not in text_clean:
                continue

            # проверяем контекст
            kw_pos = text_clean.find(kw)
            start = max(0, kw_pos - 25)
            context = text_clean[start:kw_pos]

            # отрицание в контексте
            has_negation = any(neg in context.split() for neg in negation_words)
            if not has_negation and 'не' in context:
                words_before = context.split()
                if 'не' in words_before[-2:] if len(words_before) >= 2 else words_before:
                    has_negation = True
            # если отрицание все же относится к рассматриваемой фразе, не отмечаем аспект
            if not has_negation:
                found = True
                break

        labels[aspect] = int(found)

    return labels

In [ ]:
for aspect in ASPECT_KEYWORDS.keys():
    df[aspect] = 0

for aspect in ASPECT_KEYWORDS.keys():
    df[aspect] = df['clean_text'].progress_apply(
        lambda x: weak_label_on_clean(x, ASPECT_KEYWORDS)[aspect]
    )

In [ ]:
for aspect in ASPECT_KEYWORDS.keys():
    count = df[aspect].sum()
    print(f'{aspect}: {count} ({round(count/len(df), 2)*100}%)')


In [ ]:
test_sample = df.sample(10)

for idx, row in test_sample.iterrows():
    print(f'rating: {row['rate'] if 'rate' in df.columns else 'no'}')
    print(f'row text: {row['comment'][:300]}')
    print(f'clean text: {row['clean_text'][:200]}')
    active = [aspect for aspect in ASPECT_KEYWORDS.keys() if row[aspect] == 1]
    print(f'aspects: {active if active else 'no aspects'}')

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df[list(ASPECT_KEYWORDS.keys())]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=777
)

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
df.to_csv('labeled_reviews.csv', index=False)